1. Load the three tables

In [1]:
import pandas as pd

hosts = pd.read_csv("../data/raw/RAW_HOST.csv")
listings = pd.read_csv("../data/raw/RAW_LISTING.csv")
reviews = pd.read_csv("../data/raw/RAW_REVIEWS.csv")

for name, df in [("hosts", hosts), ("listings", listings), ("reviews", reviews)]:
    print(name, df.shape)

hosts (14111, 5)
listings (17499, 9)
reviews (410284, 5)


2. Missing values and unique IDs

In [2]:
for name, df in [("hosts", hosts), ("listings", listings), ("reviews", reviews)]:
    print(f"\n--- {name} ---")
    print(df.isna().sum())

print("\nHost IDs unique:", hosts["ID"].is_unique)
print("Listing IDs unique:", listings["ID"].is_unique)
print("Exact duplicate reviews:", reviews.duplicated().sum())


--- hosts ---
ID               0
NAME            15
IS_SUPERHOST    15
CREATED_AT       0
UPDATED_AT       0
dtype: int64

--- listings ---
ID                0
LISTING_URL       0
NAME              0
ROOM_TYPE         0
MINIMUM_NIGHTS    0
HOST_ID           0
PRICE             0
CREATED_AT        0
UPDATED_AT        0
dtype: int64

--- reviews ---
LISTING_ID         0
DATE               0
REVIEWER_NAME      0
COMMENTS         592
SENTIMENT        385
dtype: int64

Host IDs unique: True
Listing IDs unique: True
Exact duplicate reviews: 1


3. Do tables join up?

In [3]:
print("Listings whose host is missing:", (~listings["HOST_ID"].isin(hosts["ID"])).sum())
print("Reviews whose listing is missing:", (~reviews["LISTING_ID"].isin(listings["ID"])).sum())
print("Listings with no reviews:", (~listings["ID"].isin(reviews["LISTING_ID"])).sum())

Listings whose host is missing: 0
Reviews whose listing is missing: 0
Listings with no reviews: 3254


4. Prices, minimum nights and room types

In [4]:
price = listings["PRICE"].str.replace(r"[$,]", "", regex=True).astype(float)
print(price.describe())
print("Price = 0:", (price == 0).sum(), "| Price > 1000:", (price > 1000).sum())

print(listings["MINIMUM_NIGHTS"].describe())
print("Min nights = 0:", (listings["MINIMUM_NIGHTS"] == 0).sum(),
      "| Over 365:", (listings["MINIMUM_NIGHTS"] > 365).sum())

print(listings["ROOM_TYPE"].value_counts())

count    17499.000000
mean        74.908623
std        114.237300
min          0.000000
25%         35.000000
50%         55.000000
75%         85.000000
max       8000.000000
Name: PRICE, dtype: float64
Price = 0: 8 | Price > 1000: 17
count    17499.000000
mean        10.035431
std         35.058413
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max       1124.000000
Name: MINIMUM_NIGHTS, dtype: float64
Min nights = 0: 82 | Over 365: 13
ROOM_TYPE
Entire home/apt    9890
Private room       7233
Shared room         211
Hotel room          165
Name: count, dtype: int64


5. Reviews over time and the sentiment label

In [5]:
reviews["DATE"] = pd.to_datetime(reviews["DATE"])
print(reviews["DATE"].dt.year.value_counts().sort_index())
print(reviews["SENTIMENT"].value_counts(dropna=False, normalize=True).round(3))
print("Automated cancellation notices:",
      reviews["COMMENTS"].str.contains("automated posting", case=False, na=False).sum())

DATE
2009         3
2010        73
2011       317
2012      1652
2013      4219
2014      9754
2015     21358
2016     35166
2017     53521
2018     77901
2019    108368
2020     50595
2021     47357
Name: count, dtype: int64
SENTIMENT
positive    0.567
neutral     0.359
negative    0.073
NaN         0.001
Name: proportion, dtype: float64
Automated cancellation notices: 4897


6. Read the reviews

In [6]:
pd.set_option("display.max_colwidth", 300)
reviews.sample(20, random_state=1)[["COMMENTS", "SENTIMENT"]]

,COMMENTS,SENTIMENT
364711,"Dagmar was a wonderful host. She picked us up from the airport and went to great trouble to make us feel at home, even looking out bath products so we could enjoy the massive bath tub",positive
314634,"I would definitely recommend staying in this hostel. The stuff was very friendly, very helpful (about the hostel, directions, places to go), the rooms and facilities were clean, the sheets alsoYou can have a drink directly in the reception area, have breakfastGreat location, we almost did not ha...",positive
35847,Great location and phenomenal response to questions and messages.,positive
51362,Ich wurde von Katarinas Mutter herzlich empfangen. Mein Zimmer war genau wie auch das Badezimmer sauber. Insgesamt war es ein angenehmer Aufenthalt.,negative
54276,"Comfortable and clean apartment, good location with access to restaurants and public transportation",positive
370756,The host canceled this reservation 26 days before arrival. This is an automated posting.,neutral
179856,"Wow This apartment is amazing. brbrThis is the only apartment weve ever stayed that truly felt like a home. Absolutely everything we needed was provided, and the hosts were always on call for tips and advice. brbrEvery inch of the space was stylish, hip and aspirational... we really felt like Be...",positive
166072,"Very nice, confortable apartment, in an extremely centric and easytoreach part of Berlin.",positive
326440,"Amazing spot. Huge cool room in the perfect location in Berlin in my opinion. Great to explore, you can walk to Boxhagener plats which has fantastic food, bars and charming shops. Close by everything you need, really loved staying there.",positive
77160,The perfect place to stay in,positive
